# Audit of the bilateral deviation labeller

The main outcome uses the previous completed calendar month's trading volume and event-day high/low. The constant 10 is provisional until source units and definitions are verified. Labels indicate volume-scaled abnormal deviations, not necessarily economically significant depegs. Counts are positive days, not episodes.

Run from a repository containing `dataset/prices.csv` and the companion Python script. This notebook rebuilds labels in memory and writes audit outputs separately. It does not run regressions or claim the mechanical-volume concern is resolved.


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'dataset' / 'prices.csv').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Open this notebook from inside the project.')
for folder in [ROOT / 'depeg_labeller', ROOT / 'regime_classifier', ROOT / 'logit']:
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from dynamic_depeg import load_prices, build_labels, monthly_volumes, breach

if not (ROOT / 'dataset/prices.csv').exists():
    raise FileNotFoundError('Set ROOT to the repository containing dataset/prices.csv.')
OUT = ROOT / 'depeg_labeller/results'
OUT.mkdir(parents=True, exist_ok=True)
prices = load_prices(ROOT / 'dataset/prices.csv')
labels, events, monthly = build_labels(prices)
print('Recorded price sources (not proof of volume provenance):')
if 'price_source' in prices:
    display(prices.groupby(['coin', 'price_source']).size().rename('rows'))
else:
    print('No price_source field.')
display(monthly)
monthly.to_csv(OUT / 'monthly_volume_audit.csv', index=False)


## Timing, coverage, and sample alignment
`date` is predictor day t; `event_date` is t+1. At a month boundary, the month ending at t is assumed available after t closes. If forecasting before that close or if vendor publication is delayed, impose a corresponding information lag. Daily timestamps must have comparable reporting boundaries.

Incomplete months are excluded; missing months are never replaced with an earlier available month. Static labels can exist without volume, but comparisons below use a shared sample. The original benchmark reproduces 30 observed rows through t, not necessarily 30 calendar days.


In [ ]:
assert (labels.event_date - labels.date).eq(pd.Timedelta(days=1)).all()
assert (labels.volume_month == labels.event_date.dt.to_period('M') - 1).all()
display(labels[['coin', 'date', 'event_date', 'volume_month',
                'previous_month_volume_usd']].head(12))
display(events.groupby('coin').agg(
    event_rows=('event_date', 'size'),
    dynamic_eligible=('eligible_dynamic', 'sum'),
    common_sample=('common_sample', 'sum')))
common = events.loc[events.common_sample].copy()
if common.empty:
    raise ValueError('No common sample; inspect monthly coverage and price validity.')
COLS = ['depeg_original_30obs', 'depeg_dynamic', 'depeg_static_25bp']
summary = common.groupby('coin')[COLS].agg(['count', 'sum', 'mean'])
display(summary)
summary.to_csv(OUT / 'common_sample_label_summary.csv')
# Use these outcomes joined to identical predictors for regression sensitivity.
common.to_csv(OUT / 'regression_label_common_sample.csv', index=False)


## Implied bands and event-day prices
Red markers identify actual high/low breaches. The second price panel zooms to ±50 bp so large events do not hide small deviations; the full-range panel retains all extremes.

In [ ]:
display(common.groupby('coin')[['threshold_bp', 'original_threshold_bp']].describe())
for coin, g in common.groupby('coin'):
    g = g.sort_values('event_date')
    fig, axes = plt.subplots(3, 1, figsize=(12, 11), sharex=True)
    axes[0].plot(g.event_date, g.original_threshold_bp, label='Original: 30 observations through t', lw=.8)
    axes[0].plot(g.event_date, g.threshold_bp, label='Previous calendar month', lw=1)
    axes[0].axhline(25, color='gray', ls='--', label='Static 25 bp')
    axes[0].set(title=coin, ylabel='Half-width (bp)')
    axes[0].legend()
    for ax in axes[1:]:
        ax.fill_between(g.event_date, g.next_day_low, g.next_day_high,
                        alpha=.25, label='Daily low–high')
        if 'next_day_close' in g:
            ax.plot(g.event_date, g.next_day_close, lw=.7, label='Close')
        ax.plot(g.event_date, g.threshold_down, color='orange', ls='--', label='Dynamic band')
        ax.plot(g.event_date, g.threshold_up, color='orange', ls='--')
        ax.axhline(.9975, color='gray', ls=':', label='Static band')
        ax.axhline(1.0025, color='gray', ls=':')
        for side, bound, op in [('next_day_low', 'threshold_down', 'le'),
                                ('next_day_high', 'threshold_up', 'ge')]:
            hit = getattr(g[side], op)(g[bound])
            ax.scatter(g.loc[hit, 'event_date'], g.loc[hit, side], color='red', s=12, zorder=3)
        ax.set_ylabel('USD')
    axes[1].legend(fontsize=8)
    axes[1].set_title('Full range')
    axes[2].set(ylim=(.995, 1.005), title='Detail: ±50 bp (larger deviations outside view)')
    fig.tight_layout()
    fig.savefig(OUT / f'{coin}_label_audit.png', dpi=160)
    plt.show()
    plt.close(fig)


## Magnitude and closing-price audit
Close is diagnostic only. Missing or inconsistent close values remain unknown and are excluded from closing-price percentages, not counted as reversals. A close inside the band says nothing about the duration of an intraday breach.

In [ ]:
common['max_intraday_deviation_bp'] = np.maximum(
    (1 - common.next_day_low).clip(lower=0),
    (common.next_day_high - 1).clip(lower=0)) * 10000
positives = common.loc[common.depeg_dynamic.eq(1)].copy()
positives['at_most_20bp'] = positives.max_intraday_deviation_bp.le(20)
positives['close_inside_band'] = pd.Series(pd.NA, index=positives.index, dtype='boolean')
if 'next_day_close' in positives:
    valid_close = (np.isfinite(positives.next_day_close)
                   & positives.next_day_close.between(positives.next_day_low, positives.next_day_high))
    positives.loc[valid_close, 'close_inside_band'] = (
        positives.loc[valid_close, 'next_day_close'].gt(positives.loc[valid_close, 'threshold_down'])
        & positives.loc[valid_close, 'next_day_close'].lt(positives.loc[valid_close, 'threshold_up']))
positives['small_intraday_only'] = pd.Series(pd.NA, index=positives.index, dtype='boolean')
known = positives.close_inside_band.notna()
positives.loc[known, 'small_intraday_only'] = (
    positives.loc[known, 'at_most_20bp'] & positives.loc[known, 'close_inside_band'])
display(positives.groupby('coin').max_intraday_deviation_bp.describe())
display(positives.groupby('coin').agg(
    positive_days=('depeg_dynamic', 'size'),
    small_share=('at_most_20bp', 'mean'),
    valid_close_days=('close_inside_band', 'count'),
    close_inside_share=('close_inside_band', 'mean'),
    small_and_close_inside_share=('small_intraday_only', 'mean')))
positives.to_csv(OUT / 'positive_observations.csv', index=False)
display(positives.nlargest(20, 'max_intraday_deviation_bp')[[
    'coin', 'event_date', 'next_day_low', 'next_day_high',
    'threshold_bp', 'max_intraday_deviation_bp']])


## CoinMarketCap volume audit (requires real reference data)
Supply `dataset/cmc_prices.csv` with `coin,date,volume_usd` and optionally `low,high,close`. Do not rename another vendor's data to create this file. Document the export origin, USD units, whether volume is daily or trailing 24-hour volume, aggregation method, reporting boundaries, and retrieval date below. Summing daily snapshots of trailing 24-hour volume may differ from exchange transaction totals; check the paper's actual measure.

This comparison holds Yahoo prices fixed and substitutes only the volume denominator. It therefore isolates denominator sensitivity. Optional price comparisons are reported separately. Agreement does not by itself establish correctness. Missing CMC data leaves this audit explicitly incomplete.


In [ ]:
CMC_PATH = ROOT / 'dataset/cmc_prices.csv'
# Fill these from source documentation, not assumptions.
VOLUME_PROVENANCE = {
    'yahoo_volume_origin_and_transformations': 'UNVERIFIED',
    'cmc_export_origin_and_retrieval_date': 'UNVERIFIED',
    'units_and_reporting_boundaries_comparable': 'UNVERIFIED',
}
display(pd.Series(VOLUME_PROVENANCE))
if not CMC_PATH.exists():
    print('NOT COMPLETED: provide a genuine CoinMarketCap export and verify definitions.')
else:
    # Reuse volume validation; reference data need not contain price columns.
    cmc = pd.read_csv(CMC_PATH)
    for name in ['low', 'high']:
        if name not in cmc:
            cmc[name] = np.nan
    # Validate with the same loader without modifying the reference file.
    cmc['coin'] = cmc.coin.astype(str).str.strip().str.upper()
    cmc = cmc[cmc.coin.isin(prices.coin.unique())].copy()
    cmc['date'] = pd.to_datetime(cmc.date, errors='raise')
    if cmc.date.dt.tz is not None or not cmc.date.eq(cmc.date.dt.normalize()).all():
        raise ValueError('Align CMC reporting dates explicitly before comparison.')
    if cmc.duplicated(['coin', 'date']).any():
        raise ValueError('Duplicate CMC coin/date rows.')
    cmc['volume_usd'] = pd.to_numeric(cmc.volume_usd, errors='raise')
    cm = monthly_volumes(cmc)
    a = monthly.loc[monthly.complete, ['coin', 'month', 'monthly_volume_usd']]
    b = cm.loc[cm.complete, ['coin', 'month', 'monthly_volume_usd']]
    matched = a.merge(b, on=['coin', 'month'], suffixes=('_yahoo', '_cmc'), validate='one_to_one')
    matched['volume_ratio_yahoo_to_cmc'] = matched.monthly_volume_usd_yahoo / matched.monthly_volume_usd_cmc
    matched['yahoo_band_bp'] = 100000 / matched.monthly_volume_usd_yahoo.pow(1/3)
    matched['cmc_band_bp'] = 100000 / matched.monthly_volume_usd_cmc.pow(1/3)
    display(matched)
    matched.to_csv(OUT / 'source_monthly_comparison.csv', index=False)
    ref = matched[['coin', 'month', 'cmc_band_bp']].copy()
    ref['month'] = ref.month + 1
    paired = common.merge(ref, on=['coin', 'month'], how='inner', validate='many_to_one')
    paired['label_cmc_volume_yahoo_prices'] = breach(
        paired.next_day_low, paired.next_day_high, paired.cmc_band_bp / 10000,
        pd.Series(True, index=paired.index))
    paired['label_disagreement'] = paired.depeg_dynamic.ne(paired.label_cmc_volume_yahoo_prices)
    display(paired.groupby('coin').agg(
        matched_events=('event_date', 'size'),
        yahoo_volume_positive_rate=('depeg_dynamic', 'mean'),
        cmc_volume_positive_rate=('label_cmc_volume_yahoo_prices', 'mean'),
        disagreement_rate=('label_disagreement', 'mean')))
    paired.to_csv(OUT / 'source_label_comparison.csv', index=False)
    daily = prices.merge(cmc, on=['coin', 'date'], suffixes=('_yahoo', '_cmc'), validate='one_to_one')
    for field in ['low', 'high', 'close']:
        if f'{field}_cmc' in daily and f'{field}_yahoo' in daily:
            daily[f'{field}_difference_bp'] = (
                pd.to_numeric(daily[f'{field}_yahoo']) - pd.to_numeric(daily[f'{field}_cmc'])) * 10000
    differences = [c for c in daily if c.endswith('_difference_bp')]
    if differences:
        display(daily.groupby('coin')[differences].agg(['count', 'median', 'min', 'max']))
    daily.to_csv(OUT / 'source_daily_comparison.csv', index=False)


## Mechanical-volume concern: remaining empirical work
Use `depeg_labeller/results/regression_label_common_sample.csv` to join the SAME predictors on `coin,date` and fit the SAME model with each of the three outcome columns. Use identical missing-predictor exclusions across models. Compare coefficient estimates, uncertainty, and sample sizes. Raw coefficients can change with outcome prevalence, so also compare average marginal effects where appropriate.

The original outcome depends on trading volume through predictor day t; the revised outcome depends on prior-month volume. Lagging removes same-event-month dependence but does not remove the volume-based definition or establish causality. Exchange trading volume and on-chain predictor volume are distinct measures that may be related. Static labels remove trading volume from the outcome definition. The regressions must be rerun before drawing conclusions about the reported on-chain-volume coefficient.

Record conclusions only after examining actual results. Source audit remains unfinished until CMC data and provenance are supplied.
